In [1]:
###################################################################
#    Accuracy assessment for COLD and SCCD (parameter tunable)    #
###################################################################

# Note: please download the dataset from google drive link (https://drive.google.com/file/d/1vwGvucNvyMXHmoIoR-pgi2VgNtsxNglw/view?usp=drive_link), and unzip it.

import os
import time
import datetime as dt
import pandas as pd
import numpy as np
from os.path import join
from pyxccd import cold_detect,cold_detect_flex,sccd_detect,sccd_detect_flex
from pyxccd.utils import getcategory_cold, getcategory_sccd

# =============================
# Paths and constants
# ============================
validation_folder = os.path.join('F:/pyxccd_data/data') # point it to the data folder you downloaded and unziped 


# ===================================================
# User-defined parameters. change them as you need
# ==================================================

p_cg = 0.99        # change probability threshold
conse = 6          # consecutive observation requirement
lam = 20           # lambda (regularization / scale parameter)
method = 'SCCD'    # options: 'COLD', 'SCCD'


spectral_path = join(validation_folder, 'nafd_singlepath_spectral')
ts_seg_path = join(validation_folder, 'correction/nafd_rearrange_v2.csv')
calibr_sample_path = join(validation_folder, 'correction/nafd_allplotid_v2.csv')

low_year_bound = 1984
upper_year_bound = 2012
threshold = 0

Landsat_bandname = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'thermal']

# -----------------------------
# Reference agent categories
# -----------------------------
ts_seg = pd.read_csv(ts_seg_path)

agent_list = set(ts_seg['change_process'].dropna().unique())
agent_list.discard('Growth/Recovery')
agent_list.discard('Stable')
agent_list = sorted(agent_list)

n_agent = len(agent_list)

agentcateg_ref_count = [0] * n_agent
agentcateg_match_count = [0] * n_agent

predict_dist_count = 0
predict_match_count = 0
total_ref_count = 0

# -----------------------------
# Sample IDs
# -----------------------------
tst_ids = pd.read_csv(calibr_sample_path)['plotid'].tolist()

# -----------------------------
# Accuracy records
# -----------------------------
ccd_acc_records = pd.DataFrame()

start_time = time.time()

# ============================================================
# Main loop
# ============================================================
for count, pid in enumerate(tst_ids):
    print("Processing {} / {} plot: {}".format(count + 1, len(tst_ids), pid))

    single_match_count = 0
    single_ccd_count = 0
    single_ref_count = 0

    foc_tst_ids = ts_seg.loc[ts_seg['plotid'] == pid]

    in_path = join(spectral_path, f"spectral_{pid}.csv")
    if not os.path.exists(in_path):
        continue

    # -----------------------------
    # Load spectral data
    # -----------------------------
    data = pd.read_csv(in_path, header=None)
    data.iloc[:, 0] = data.iloc[:, 0] - 366  # MATLAB → Python ordinal date

    data.columns = ['dates'] + Landsat_bandname + ['qa', 'sensor']
    data = data[data['dates'] > threshold]

    # -----------------------------
    # Change detection
    # -----------------------------
    try:
        if method == 'COLD':
            change_records = cold_detect(
                data['dates'].to_numpy(),
                data[Landsat_bandname[0]].to_numpy(),
                data[Landsat_bandname[1]].to_numpy(),
                data[Landsat_bandname[2]].to_numpy(),
                data[Landsat_bandname[3]].to_numpy(),
                data[Landsat_bandname[4]].to_numpy(),
                data[Landsat_bandname[5]].to_numpy(),
                data[Landsat_bandname[6]].to_numpy().copy(),
                data['qa'].to_numpy(),
                p_cg=p_cg,
                conse=conse,
                b_c2=False,
                lam=lam
            )
            detectionresults = change_records

        elif method == 'SCCD':
            change_records = sccd_detect(
                data['dates'].to_numpy(),
                data[Landsat_bandname[0]].to_numpy(),
                data[Landsat_bandname[1]].to_numpy(),
                data[Landsat_bandname[2]].to_numpy(),
                data[Landsat_bandname[3]].to_numpy(),
                data[Landsat_bandname[4]].to_numpy(),
                data[Landsat_bandname[5]].to_numpy(),
                data['qa'].to_numpy(),
                p_cg=p_cg,
                conse=conse,
                b_c2=False,
                lam=lam
            )
            detectionresults = change_records.rec_cg

    except Exception as e:
        print("Processing failed for pid {}: {}".format(pid, e))

        # Count reference disturbances even if detection fails
        for _, ref_row in foc_tst_ids.iterrows():
            if ref_row['change_process'] not in agent_list:
                continue
            if ref_row['beginning'] == low_year_bound:
                continue
            total_ref_count += 1
            single_ref_count += 1
            agent_idx = agent_list.index(ref_row['change_process'])
            agentcateg_ref_count[agent_idx] += 1

        acc_row = [pid, single_match_count, single_ccd_count, single_ref_count]
        ccd_acc_records = pd.concat(
            [ccd_acc_records, pd.DataFrame([acc_row])],
            ignore_index=True
        )
        continue

    # -----------------------------
    # Evaluate detections
    # -----------------------------
    ref_used_tag = [False] * len(foc_tst_ids)

    for i, curve in enumerate(detectionresults):
        if curve['t_break'] == 0:
            continue

        if method == 'COLD' and curve['change_prob'] < 100:
            continue

        if method == 'COLD':
            if i == len(detectionresults) - 1:
                continue
            if getcategory_cold(detectionresults, i) == 2:
                continue
        else:  # SCCD
            if getcategory_sccd(detectionresults, i) == 2:
                continue

        break_year = pd.Timestamp.fromordinal(curve['t_break']).year
        if break_year < low_year_bound or break_year > upper_year_bound:
            continue

        for index, ref_row in foc_tst_ids.iterrows():
            if ref_row['change_process'] not in agent_list:
                continue

            if ref_row['ending'] >= break_year >= ref_row['beginning']:
                if ref_used_tag[index - foc_tst_ids.index[0]]:
                    predict_dist_count -= 1
                    single_ccd_count -= 1
                    break

                predict_match_count += 1
                single_match_count += 1
                ref_used_tag[index - foc_tst_ids.index[0]] = True
                break

        predict_dist_count += 1
        single_ccd_count += 1

    # -----------------------------
    # Count reference disturbances
    # -----------------------------
    for _, ref_row in foc_tst_ids.iterrows():
        if ref_row['change_process'] not in agent_list:
            continue
        if ref_row['beginning'] == low_year_bound:
            continue

        total_ref_count += 1
        single_ref_count += 1
        agent_idx = agent_list.index(ref_row['change_process'])
        agentcateg_ref_count[agent_idx] += 1

    acc_row = [pid, single_match_count, single_ccd_count, single_ref_count]
    ccd_acc_records = pd.concat(
        [ccd_acc_records, pd.DataFrame([acc_row])],
        ignore_index=True
    )

# ============================================================
# Final metrics
# ============================================================
algorithm_omission_rate = (total_ref_count - predict_match_count) / total_ref_count
algorithm_commission_rate = (predict_dist_count - predict_match_count) / predict_dist_count
algorithm_F_score = (
    (1 - algorithm_commission_rate)
    * (1 - algorithm_omission_rate)
    / (2 - algorithm_omission_rate - algorithm_commission_rate)
    * 2
)

print("\n====== Final Results ======")
print("Omission rate   : {:.4f}".format(algorithm_omission_rate))
print("Commission rate : {:.4f}".format(algorithm_commission_rate))
print("F-score         : {:.4f}".format(algorithm_F_score))
print("Elapsed time    : {:.2f} seconds".format(time.time() - start_time))

Processing 1 / 6488 plot: 23036001
Processing 2 / 6488 plot: 23036002
Processing 3 / 6488 plot: 23036003
Processing 4 / 6488 plot: 23036004
Processing 5 / 6488 plot: 23036005
Processing 6 / 6488 plot: 23036007
Processing 7 / 6488 plot: 23036008
Processing 8 / 6488 plot: 23036009
Processing 9 / 6488 plot: 23036010
Processing 10 / 6488 plot: 23036011
Processing 11 / 6488 plot: 23036012
Processing 12 / 6488 plot: 23036013
Processing 13 / 6488 plot: 23036014
Processing 14 / 6488 plot: 23036015
Processing 15 / 6488 plot: 23036016
Processing 16 / 6488 plot: 23036017
Processing 17 / 6488 plot: 23036018
Processing 18 / 6488 plot: 23036019
Processing 19 / 6488 plot: 23036020
Processing 20 / 6488 plot: 23036021
Processing 21 / 6488 plot: 23036022
Processing 22 / 6488 plot: 23036023
Processing 23 / 6488 plot: 23036024
Processing 24 / 6488 plot: 23036025
Processing 25 / 6488 plot: 23036026
Processing 26 / 6488 plot: 23036027
Processing 27 / 6488 plot: 23036028
Processing 28 / 6488 plot: 23036029
P

In [ ]:
#######################################################################
#     Efficiency benchmark for COLD algorithm (retrospective & NRT)   #
#######################################################################

import datetime as dt
import time
import os
import pandas as pd
from os.path import join
import numpy as np
from pyxccd import cold_detect_flex

# =======================================================================
# User-configurable section (edit here)
# =======================================================================
validation_folder = os.path.join('E:/paper_figures/data')
spectral_path = join(validation_folder, 'nafd_singlepath_spectral')
calibr_sample_path = join(validation_folder, 'valid_sample_ids_5000.csv')

# Observation mode:
#   "NRT"  -> use the most recent 300 observations
#   "retrospective"-> use the most recent 290 observations (exclude last 10)
mode = "NRT"

# =======================================================================
# Constants
# =======================================================================
Landsat_bandname = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'thermal']
threshold = 0

# Read test sample IDs
tst_ids = pd.read_csv(calibr_sample_path)['plotid'].tolist()

print(f"Start benchmarking, total samples: {len(tst_ids)}")
print(f"Observation mode: {mode}")
print("-" * 60)

# Define 10 benchmark configurations
tests_config = []

# Add 10 COLDF tests
for i in range(1, 11):
    config = {
        'method': 'COLDF',
        'band_count': i,
        'band_indices': []
    }

    # Determine band indices
    if i <= 5:
        config['band_indices'] = [1, 2, 3, 4, 5][:i]
    else:
        base_indices = [1, 2, 3, 4, 5]
        repeated_indices = []
        while len(repeated_indices) < i:
            repeated_indices.extend(base_indices)
        config['band_indices'] = repeated_indices[:i]

    tests_config.append(config)

# Store all benchmark results
all_results = []

# Iterate over all test configurations
for test_num, config in enumerate(tests_config, 1):
    method = config['method']
    band_count = config['band_count']
    band_indices = config['band_indices']

    print(f"\nRunning test {test_num}/10: {method} with {band_count} bands")
    print(f"Band indices: {band_indices}")

    core_algorithm_time_total = 0.0
    core_algorithm_count = 0
    processing_times = []

    start_total_time = time.time()

    for count, pid in enumerate(tst_ids):
        in_path = join(spectral_path, f"spectral_{pid}.csv")

        if not os.path.exists(in_path):
            continue

        try:
            # Load spectral data
            data = pd.read_csv(in_path, header=None)
            data.iloc[:, 0] = data.iloc[:, 0] - 366  # MATLAB → Python date

            data.columns = ['dates'] + Landsat_bandname + ['qa', 'sensor']
            data = data[data['dates'] > threshold]
            data = data[data['qa'] == 0]

            # -----------------------------------------------------------
            # Observation selection based on mode
            # -----------------------------------------------------------
            if mode == "NRT":
                n_use = min(len(data), 300)
                data_used = data.iloc[-n_use:]

            elif mode == "retrospective":
                if len(data) < 11:
                    continue
                n_use = min(len(data) - 10, 290)
                data_used = data.iloc[-(n_use + 10):-10]

            else:
                raise ValueError("Invalid mode. Use 'NRT' or 'retrospective'.")

            # Prepare inputs
            dates_used = data_used['dates'].to_numpy()

            band_data = []
            for idx in band_indices:
                band_data.append(data_used[Landsat_bandname[idx]].to_numpy())

            band_used = np.stack(band_data, axis=1)
            qa_used = data_used['qa'].to_numpy()

            # Core algorithm timing
            algorithm_start_time = time.time()

            _ = cold_detect_flex(
                dates_used.astype(np.int64, order="C"),
                band_used.astype(np.int64, order="C"),
                qa_used.astype(np.int64, order="C"),
                20
            )

            algorithm_end_time = time.time()
            algorithm_time = algorithm_end_time - algorithm_start_time

            core_algorithm_time_total += algorithm_time
            core_algorithm_count += 1
            processing_times.append(algorithm_time)

        except Exception as e:
            raise

    end_total_time = time.time()
    total_elapsed_time = end_total_time - start_total_time

    result = {
        'test_id': test_num,
        'method': method,
        'band_count': band_count,
        'core_algorithm_time_sec': core_algorithm_time_total,
        'total_time_sec': total_elapsed_time,
        'successful_samples': core_algorithm_count
    }

    all_results.append(result)

    print(f"  Finished: core algorithm total time = {core_algorithm_time_total:.4f} seconds")

# =======================================================================
# Summary output
# =======================================================================
print("\n" + "=" * 80)
print("Benchmark Results Summary (10 Tests)")
print("=" * 80)
print(f"{'Test ID':<8} {'Method':<8} {'Bands':<8} "
      f"{'Core Algorithm Time (s)':<22} {'Total Time (s)':<16} {'Samples':<10}")
print("-" * 80)

for r in all_results:
    print(f"{r['test_id']:<8} {r['method']:<8} {r['band_count']:<8} "
          f"{r['core_algorithm_time_sec']:<22.4f} {r['total_time_sec']:<16.4f} "
          f"{r['successful_samples']:<10}")

print("=" * 80)

# Save results
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
results_df = pd.DataFrame(all_results)
csv_file = f"cold_detect_flex_benchmark_{mode}_{timestamp}.csv"
results_df.to_csv(csv_file, index=False)

txt_file = f"cold_detect_flex_benchmark_summary_{mode}_{timestamp}.txt"
with open(txt_file, 'w', encoding='utf-8') as f:
    f.write(f"Benchmark Results Summary (Mode: {mode})\n")
    f.write("=" * 80 + "\n")
    f.write(f"{'Test ID':<8} {'Method':<8} {'Bands':<8} "
            f"{'Core Algorithm Time (s)':<22} {'Total Time (s)':<16} {'Samples':<10}\n")
    f.write("-" * 80 + "\n")
    for r in all_results:
        f.write(f"{r['test_id']:<8} {r['method']:<8} {r['band_count']:<8} "
                f"{r['core_algorithm_time_sec']:<22.4f} {r['total_time_sec']:<16.4f} "
                f"{r['successful_samples']:<10}\n")
    f.write("=" * 80 + "\n")
    f.write(f"\nBenchmark completed at: {dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"\nDetailed results saved to CSV: {csv_file}")
print(f"Summary saved to text file: {txt_file}")

# Key results
print("\n" + "=" * 80)
print("Key Results (Core Algorithm Time vs Number of Bands)")
print("=" * 80)
print(f"{'Test ID':<8} {'Method':<8} {'Bands':<8} {'Core Algorithm Time (s)':<22}")
print("-" * 80)
for r in all_results:
    print(f"{r['test_id']:<8} {r['method']:<8} {r['band_count']:<8} "
          f"{r['core_algorithm_time_sec']:<22.4f}")
print("=" * 80)

In [ ]:
########################################################
#    Test the efficiency for S-CCD (retrospective)     #
########################################################

import datetime as dt
import time
import os
import pandas as pd
import numpy as np
from os.path import join
from pyxccd import sccd_detect_flex

# =======================================================================
# User-configurable paths (edit here)
# =======================================================================
validation_folder = os.path.join('E:/paper_figures/data')
spectral_path = join(validation_folder, 'nafd_singlepath_spectral')
calibr_sample_path = join(validation_folder, 'valid_sample_ids_5000.csv')

# =======================================================================
# Constants
# =======================================================================
Landsat_bandname = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'thermal']
threshold = 0

# Read test sample IDs
tst_ids = pd.read_csv(calibr_sample_path)['plotid'].tolist()

print(f"Start benchmarking, total samples: {len(tst_ids)}")
print("-" * 60)

# Define 10 benchmark configurations
tests_config = []

# Add 10 SCCDF tests
for i in range(1, 11):
    config = {
        'method': 'SCCDF',
        'band_count': i,
        'band_indices': []
    }

    if i <= 5:
        config['band_indices'] = [1, 2, 3, 4, 5][:i]
    else:
        base_indices = [1, 2, 3, 4, 5]
        repeated_indices = []
        while len(repeated_indices) < i:
            repeated_indices.extend(base_indices)
        config['band_indices'] = repeated_indices[:i]

    tests_config.append(config)

# Store all benchmark results
all_results = []

# Iterate over all test configurations
for test_num, config in enumerate(tests_config, 1):
    method = config['method']
    band_count = config['band_count']
    band_indices = config['band_indices']

    print(f"\nRunning test {test_num}/10: {method} with {band_count} bands")
    print(f"Band indices: {band_indices}")

    core_algorithm_time_total = 0.0
    core_algorithm_count = 0
    processing_times = []

    start_total_time = time.time()

    for count, pid in enumerate(tst_ids):
        in_path = join(spectral_path, f"spectral_{pid}.csv")

        if not os.path.exists(in_path):
            continue

        try:
            # Load spectral data
            data = pd.read_csv(in_path, header=None)
            data.iloc[:, 0] = data.iloc[:, 0] - 366  # MATLAB → Python date

            data.columns = ['dates'] + Landsat_bandname + ['qa', 'sensor']
            data = data[data['dates'] > threshold]
            data = data[data['qa'] == 0]

            # Use only the most recent 300 observations
            n_use = min(len(data), 300)
            data_used = data.iloc[-n_use:]

            # Prepare inputs
            dates_used = data_used['dates'].to_numpy()

            band_data = []
            for idx in band_indices:
                band_data.append(data_used[Landsat_bandname[idx]].to_numpy())

            band_used = np.stack(band_data, axis=1)
            qa_used = data_used['qa'].to_numpy()

            # Core algorithm timing
            algorithm_start_time = time.time()

            _ = sccd_detect_flex(
                dates_used.astype(np.int64, order="C"),
                band_used.astype(np.int64, order="C"),
                qa_used.astype(np.int64, order="C"),
                20
            )

            algorithm_end_time = time.time()
            algorithm_time = algorithm_end_time - algorithm_start_time

            core_algorithm_time_total += algorithm_time
            core_algorithm_count += 1
            processing_times.append(algorithm_time)

        except Exception as e:
            raise

    end_total_time = time.time()
    total_elapsed_time = end_total_time - start_total_time

    result = {
        'test_id': test_num,
        'method': method,
        'band_count': band_count,
        'core_algorithm_time_sec': core_algorithm_time_total,
        'total_time_sec': total_elapsed_time,
        'successful_samples': core_algorithm_count
    }

    all_results.append(result)

    print(f"  Finished: core algorithm total time = {core_algorithm_time_total:.4f} seconds")

# =======================================================================
# Summary output
# =======================================================================
print("\n" + "=" * 80)
print("Benchmark Results Summary (10 Tests)")
print("=" * 80)
print(f"{'Test ID':<8} {'Method':<8} {'Bands':<8} "
      f"{'Core Algorithm Time (s)':<22} {'Total Time (s)':<16} {'Samples':<10}")
print("-" * 80)

for r in all_results:
    print(f"{r['test_id']:<8} {r['method']:<8} {r['band_count']:<8} "
          f"{r['core_algorithm_time_sec']:<22.4f} {r['total_time_sec']:<16.4f} "
          f"{r['successful_samples']:<10}")

print("=" * 80)

# Save detailed results to CSV
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
results_df = pd.DataFrame(all_results)
csv_file = f"sccd_detect_flex_benchmark_{timestamp}.csv"
results_df.to_csv(csv_file, index=False, encoding='utf-8')

# Save summary to text file
txt_file = f"sccd_detect_flex_benchmark_summary_{timestamp}.txt"
with open(txt_file, 'w', encoding='utf-8') as f:
    f.write("Benchmark Results Summary (10 Tests)\n")
    f.write("=" * 80 + "\n")
    f.write(f"{'Test ID':<8} {'Method':<8} {'Bands':<8} "
            f"{'Core Algorithm Time (s)':<22} {'Total Time (s)':<16} {'Samples':<10}\n")
    f.write("-" * 80 + "\n")

    for r in all_results:
        f.write(f"{r['test_id']:<8} {r['method']:<8} {r['band_count']:<8} "
                f"{r['core_algorithm_time_sec']:<22.4f} {r['total_time_sec']:<16.4f} "
                f"{r['successful_samples']:<10}\n")

    f.write("=" * 80 + "\n")
    f.write("\nTest Configurations:\n")
    for cfg in tests_config:
        f.write(f"Test {cfg['band_count']}: {cfg['method']} | "
                f"Bands = {cfg['band_count']} | "
                f"Band indices = {cfg['band_indices']}\n")

    f.write(f"\nBenchmark completed at: {dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"\nDetailed results saved to CSV: {csv_file}")
print(f"Summary saved to text file: {txt_file}")

# Key results (core algorithm time vs number of bands)
print("\n" + "=" * 80)
print("Key Results (Core Algorithm Time vs Number of Bands)")
print("=" * 80)
print(f"{'Test ID':<8} {'Method':<8} {'Bands':<8} {'Core Algorithm Time (s)':<22}")
print("-" * 80)
for r in all_results:
    print(f"{r['test_id']:<8} {r['method']:<8} {r['band_count']:<8} "
          f"{r['core_algorithm_time_sec']:<22.4f}")
print("=" * 80)

In [ ]:
################################################
#    Test the efficiency for S-CCD online      #
################################################

import datetime as dt
import time
import os
import pandas as pd
import numpy as np
from os.path import join
from pyxccd.ccd import sccd_update_flex

# =======================================================================
# User-configurable paths (edit here)
# =======================================================================
validation_folder = os.path.join('E:/paper_figures/data')
spectral_path = join(validation_folder, 'nafd_singlepath_spectral')
calibr_sample_path = join(validation_folder, 'valid_sample_ids_5000.csv')

# =======================================================================
# Constants
# =======================================================================
Landsat_bandname = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'thermal']
threshold = 0

# Read test sample IDs
tst_ids = pd.read_csv(calibr_sample_path)['plotid'].tolist()

print(f"Start benchmarking, total samples: {len(tst_ids)}")
print("-" * 60)

# Define 10 benchmark configurations
tests_config = []

# Run SCCD update for 1 to 10 bands
for i in range(1, 11):
    config = {
        'method': 'SCCDF_UPDATE',
        'band_count': i,
        'band_indices': []
    }

    if i <= 5:
        config['band_indices'] = [1, 2, 3, 4, 5][:i]
    else:
        base_indices = [1, 2, 3, 4, 5]
        repeated_indices = []
        while len(repeated_indices) < i:
            repeated_indices.extend(base_indices)
        config['band_indices'] = repeated_indices[:i]

    tests_config.append(config)

# Store all benchmark results
all_results = []

# Iterate over all test configurations
for test_num, config in enumerate(tests_config, 1):
    method = config['method']
    band_count = config['band_count']
    band_indices = config['band_indices']

    print(f"\nRunning test {test_num}/10: {method} with {band_count} bands")
    print(f"Band indices: {band_indices}")

    core_algorithm_time_total = 0.0
    core_algorithm_count = 0
    processing_times = []

    start_total_time = time.time()

    for count, pid in enumerate(tst_ids):
        in_path = join(spectral_path, f"spectral_{pid}.csv")

        if not os.path.exists(in_path):
            continue

        try:
            # Load spectral data
            data = pd.read_csv(in_path, header=None)
            data.iloc[:, 0] = data.iloc[:, 0] - 366  # MATLAB → Python date

            data.columns = ['dates'] + Landsat_bandname + ['qa', 'sensor']
            data = data[data['dates'] > threshold]
            data = data[data['qa'] == 0]

            # Require at least 11 valid observations
            if len(data) < 11:
                continue

            # Split historical and new observations
            n_hist = min(290, len(data) - 10)
            data_hist = data.iloc[-(n_hist + 10):-10]
            data_upd = data.iloc[-10:]

            # Historical inputs
            dates_hist = data_hist['dates'].to_numpy()
            qa_hist = data_hist['qa'].to_numpy()

            band_hist = []
            for idx in band_indices:
                band_hist.append(data_hist[Landsat_bandname[idx]].to_numpy())
            band_hist = np.stack(band_hist, axis=1)

            # Update inputs
            dates_upd = data_upd['dates'].to_numpy()
            qa_upd = data_upd['qa'].to_numpy()

            band_upd = []
            for idx in band_indices:
                band_upd.append(data_upd[Landsat_bandname[idx]].to_numpy())
            band_upd = np.stack(band_upd, axis=1)

            # Run SCCD detection (not timed)
            change_records = sccd_detect_flex(
                dates_hist.astype(np.int64, order="C"),
                band_hist.astype(np.int64, order="C"),
                qa_hist.astype(np.int64, order="C"),
                20
            )

            # Core update algorithm timing
            algorithm_start_time = time.time()

            _ = sccd_update_flex(
                change_records,
                dates_upd.astype(np.int64, order="C"),
                band_upd.astype(np.int64, order="C"),
                qa_upd.astype(np.int64, order="C"),
                20
            )

            algorithm_end_time = time.time()
            algorithm_time = algorithm_end_time - algorithm_start_time

            core_algorithm_time_total += algorithm_time
            core_algorithm_count += 1
            processing_times.append(algorithm_time)

        except Exception as e:
            raise

    end_total_time = time.time()
    total_elapsed_time = end_total_time - start_total_time

    result = {
        'test_id': test_num,
        'method': method,
        'band_count': band_count,
        'core_algorithm_time_sec': core_algorithm_time_total,
        'total_time_sec': total_elapsed_time,
        'successful_samples': core_algorithm_count
    }

    all_results.append(result)

    print(f"  Finished: core update algorithm total time = {core_algorithm_time_total:.4f} seconds")

# =======================================================================
# Summary output
# =======================================================================
print("\n" + "=" * 80)
print("Benchmark Results Summary (10 Tests)")
print("=" * 80)
print(f"{'Test ID':<8} {'Method':<12} {'Bands':<8} "
      f"{'Core Algorithm Time (s)':<22} {'Total Time (s)':<16} {'Samples':<10}")
print("-" * 80)

for r in all_results:
    print(f"{r['test_id']:<8} {r['method']:<12} {r['band_count']:<8} "
          f"{r['core_algorithm_time_sec']:<22.4f} {r['total_time_sec']:<16.4f} "
          f"{r['successful_samples']:<10}")

print("=" * 80)

# Save detailed results to CSV
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
results_df = pd.DataFrame(all_results)
csv_file = f"sccd_update_flex_benchmark_{timestamp}.csv"
results_df.to_csv(csv_file, index=False, encoding='utf-8')

# Save summary to text file
txt_file = f"sccd_update_flex_benchmark_summary_{timestamp}.txt"
with open(txt_file, 'w', encoding='utf-8') as f:
    f.write("Benchmark Results Summary (10 Tests)\n")
    f.write("=" * 80 + "\n")
    f.write(f"{'Test ID':<8} {'Method':<12} {'Bands':<8} "
            f"{'Core Algorithm Time (s)':<22} {'Total Time (s)':<16} {'Samples':<10}\n")
    f.write("-" * 80 + "\n")

    for r in all_results:
        f.write(f"{r['test_id']:<8} {r['method']:<12} {r['band_count']:<8} "
                f"{r['core_algorithm_time_sec']:<22.4f} {r['total_time_sec']:<16.4f} "
                f"{r['successful_samples']:<10}\n")

    f.write("=" * 80 + "\n")
    f.write("\nTest Configurations:\n")
    for cfg in tests_config:
        f.write(f"Test {cfg['band_count']}: {cfg['method']} | "
                f"Bands = {cfg['band_count']} | "
                f"Band indices = {cfg['band_indices']}\n")

    f.write(f"\nBenchmark completed at: {dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"\nDetailed results saved to CSV: {csv_file}")
print(f"Summary saved to text file: {txt_file}")

# Key results (core update algorithm time vs number of bands)
print("\n" + "=" * 80)
print("Key Results (Core Update Algorithm Time vs Number of Bands)")
print("=" * 80)
print(f"{'Test ID':<8} {'Method':<12} {'Bands':<8} {'Core Algorithm Time (s)':<22}")
print("-" * 80)
for r in all_results:
    print(f"{r['test_id']:<8} {r['method']:<12} {r['band_count']:<8} "
          f"{r['core_algorithm_time_sec']:<22.4f}")
print("=" * 80)